In [4]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

# 1. 读取数据
df = pd.read_csv("./data.csv")
# 提取特征和标签
X = df.iloc[:, :14].values  # 前14列流水数据
y = df.iloc[:, 14].values   # 第15列欺诈标签(0/1)
# print(y)

# 2. 转换为PyTorch张量
# 特征转换为float32类型，标签也转换为float32以匹配损失函数要求
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).view(-1, 1)  # 形状变为(n_samples, 1)

print("数据集大小：", X_tensor.shape[0])
print("特征维度：", X_tensor.shape[1])
print("标签维度：", y_tensor.shape[1])

数据集大小： 620
特征维度： 14
标签维度： 1


In [5]:
# 3. 定义模型
class FraudModel(nn.Module):
    def __init__(self):
        super(FraudModel, self).__init__()
        # 一个线性层：14维输入，1维输出
        self.linear = nn.Linear(14, 1)
        # Sigmoid激活函数，将输出映射到(0,1)区间
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        out = self.linear(x)
        out = self.sigmoid(out)
        return out

# 初始化模型
model = FraudModel()

# 4. 定义损失函数和优化器
criterion = nn.BCELoss()  # 二分类交叉熵损失函数
optimizer = optim.SGD(model.parameters(), lr=0.0001)  # 学习率0.0001

# 5. 定义训练函数
def train_model(model, X, y, criterion, optimizer, epochs=2000, batch_size=20):
    n_samples = X.shape[0]
    n_batches = n_samples // batch_size  # 计算总批次数
    
    for epoch in range(epochs):
        running_loss = 0.0
        
        # 批量训练
        for i in range(n_batches):
            # 获取当前批次数据
            start_idx = i * batch_size
            end_idx = start_idx + batch_size
            X_batch = X[start_idx:end_idx]
            y_batch = y[start_idx:end_idx]
            
            # 前向传播
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            
            # 梯度归零 + 反向传播 + 优化
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        # 每200个epoch打印一次损失
        if (epoch + 1) % 200 == 0:
            avg_loss = running_loss / n_batches
            print(f"Epoch [{epoch+1}/{epochs}], 平均损失: {avg_loss:.4f}")
    
    print("训练完成！")

# 开始训练
train_model(model, X_tensor, y_tensor, criterion, optimizer)


Epoch [200/2000], 平均损失: 15.3226
Epoch [400/2000], 平均损失: 15.3226
Epoch [600/2000], 平均损失: 15.3226
Epoch [800/2000], 平均损失: 15.3226
Epoch [1000/2000], 平均损失: 15.3226
Epoch [1200/2000], 平均损失: 15.3226
Epoch [1400/2000], 平均损失: 15.3226
Epoch [1600/2000], 平均损失: 15.3226
Epoch [1800/2000], 平均损失: 15.3226
Epoch [2000/2000], 平均损失: 15.3226
训练完成！


In [6]:
# 6. 在整个数据集上计算准确率
with torch.no_grad():  # 关闭梯度计算，提高速度
    predictions = model(X_tensor)
    # 将预测值>0.5视为1(欺诈)，否则为0(正常)
    predicted_labels = (predictions > 0.5).float()
    # 计算准确率
    accuracy = (predicted_labels == y_tensor).sum().item() / y_tensor.shape[0]
    print(f"模型在训练集上的准确率: {accuracy * 100:.2f}%")


模型在训练集上的准确率: 84.68%


In [ ]:
# 帮我生成代码实现预测银行流水欺诈。导入torch的nn包，先用pandas提取出data.csv的数据，
# 再定义X=前十四个流水数据，Y=第十五个是否为欺诈流水。再将数据转换为tensor（张量）。
# 再定义模型为一个线性层，14维度输入，1维度输出，使用sigmoid函数。再定义优化器，lr=0.0001。
# 再定义训练函数，循环训练2000次，每批次训练20个，得到所有的预测结果，再用损失函数计算损失，
# 然后梯度归零，反向传播，使用优化器。最后看看训练预测的准确率（重新预测data.csv看看准确率）